# 07 — Chennai: zero-shot domain transfer

Applies the OSCD-trained model, **unchanged**, to Sentinel-2 scenes over Chennai
acquired independently via Google Earth Engine — no Chennai labels, no fine-tuning.

Three epochs (**Jan–Mar of 2018, 2021, 2025** — all post-monsoon) answer two questions:
- **What changed?** a full change-probability map for the 2018 → 2025 span;
- **When did it change?** pairwise maps (2018 → 2021, 2021 → 2025) that localize
  each change to the period it appeared in.

Deliverables:
1. a reusable Sentinel-2 L2A ingestion path for an arbitrary AOI (GEE acquisition + lazy GeoTIFF reader),
2. georeferenced change-probability GeoTIFFs for the full Chennai metro + fringe,
3. presentation-ready figures (real dates on every panel, no jargon) + a full reproducibility record.

OSCD-derived labels exist only for the 24 OSCD cities, so the Chennai output is evaluated
qualitatively — consistent with a zero-shot transfer study; OSM-derived labels are future work.

**One-time setup (outside Colab):** register a free noncommercial Earth Engine project at
https://code.earthengine.google.com/register and note its project ID (needed in the auth cell below).

**Storage contract** — code lives on GitHub (branch `chennai`), data & checkpoints live on Drive
(`MyDrive/s2cd`), `/content` is disposable scratch. Run all cells top-to-bottom in a fresh GPU session.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import os

REPO_URL = "https://github.com/Aadith-Geeth-Mohan/sentinel2-change-detection.git"
REPO = "/content/s2-change-detection"
BRANCH = "chennai"          # Chennai phase lives on this branch
if os.path.exists(REPO):
    !git -C {REPO} fetch origin
    !git -C {REPO} checkout {BRANCH}
    !git -C {REPO} pull --ff-only origin {BRANCH}
else:
    !git clone -b {BRANCH} {REPO_URL} {REPO}
%cd {REPO}
!pip install -q .
import sys
sys.path.insert(0, os.path.join(REPO, "src"))
# drop cached s2cd modules so a just-pulled fix takes effect in THIS kernel
for k in [k for k in list(sys.modules) if k == "s2cd" or k.startswith("s2cd.")]:
    del sys.modules[k]
# tripwire: s2_l2a.py exists only on the chennai branch — a failed checkout
# (branch not pushed yet?) must not be silently worked around
assert os.path.exists(os.path.join(REPO, "src/s2cd/data/s2_l2a.py")), \
    f"s2_l2a.py missing — is branch '{BRANCH}' pushed to origin?"
print("chennai branch checked out")


In [ ]:
import torch
from s2cd.data import DriveStore
from s2cd.utils import get_device, describe_device

device = get_device()
assert device.type == "cuda", "Runtime > Change runtime type > select a GPU"
print(describe_device(device))
store = DriveStore()  # Drive root: MyDrive/s2cd · scratch: /content


## Acquire the three Sentinel-2 scenes from Earth Engine

All three epochs use the same post-monsoon window (Jan–Mar): clearest skies over Chennai, and
same-season pairing suppresses spurious 'change' from vegetation phenology. For each epoch the
least-cloudy scene is exported with the SAME 10 bands the OSCD pipeline uses — native 10 m bands
plus the 20 m bands bilinearly resampled at export time, mirroring `oscd.py`'s preprocessing —
as single uint16 GeoTIFFs to Drive.

In [ ]:
!pip install -q earthengine-api
import ee
ee.Authenticate()   # opens a browser tab for the one-time OAuth consent

EE_PROJECT = "your-gee-project-id"  # TODO: from code.earthengine.google.com/register
assert EE_PROJECT != "your-gee-project-id", "set EE_PROJECT to your registered GEE project id"
ee.Initialize(project=EE_PROJECT)
print("Earth Engine initialized:", EE_PROJECT)


In [ ]:
from s2cd.data.gee_acquire import CHENNAI_RECT, find_scene, list_scenes

EPOCHS = {"2018": ("2018-01-01", "2018-03-31"),   # post-monsoon windows
          "2021": ("2021-01-01", "2021-03-31"),
          "2025": ("2025-01-01", "2025-03-31")}
CLOUD_MAX = 5.0

print("AOI [W, S, E, N]:", CHENNAI_RECT)
scene_info = {}
for year, (d0, d1) in EPOCHS.items():
    print(f"\ncandidate scenes for {year} ({d0}..{d1}):")
    for s in list_scenes(CHENNAI_RECT, d0, d1, cloud_max=CLOUD_MAX, limit=5):
        print(f"  {s['date']}  cloud {s['cloud_pct']:4.1f}%  {s['id']}")
    scene_info[year] = find_scene(CHENNAI_RECT, d0, d1, cloud_max=CLOUD_MAX)
    print(f"  -> selected {scene_info[year]['date']} "
          f"(cloud {scene_info[year]['cloud_pct']}%)")


In [ ]:
from s2cd.data.gee_acquire import export_scene, wait_for_tasks

# ~1.2 GB per date as uint16 -> single-file Drive exports (EE would split
# larger exports into chunks; load_scene stitches those if the AOI grows)
tasks = [export_scene(scene_info[year]["image"], CHENNAI_RECT, f"chennai_{year}")
         for year in EPOCHS]
wait_for_tasks(tasks)   # polls every 30 s; typically 5-15 min per export


## Stage & lazy-load the scenes

`L2AScene` reads only the window `predict_city` asks for — the full 3-date × 10-band stack
never materializes in RAM. It duck-types the `read_city` ndarray contract (`.shape` +
window slices), so `evaluate.predict_city` runs unchanged.

In [ ]:
import shutil
from pathlib import Path
from s2cd.data import load_scene

DRIVE_CH = store.drive / "chennai"       # export folder was s2cd/chennai
stage = Path("/content/chennai"); stage.mkdir(exist_ok=True)
for year in EPOCHS:
    src = DRIVE_CH / f"chennai_{year}.tif"
    assert src.exists(), f"{src} missing — did the export cell complete?"
    shutil.copy(src, stage / f"chennai_{year}.tif")

scenes = {year: load_scene(stage / f"chennai_{year}.tif") for year in EPOCHS}
ref_shape = scenes["2018"].shape
for year, s in scenes.items():
    assert s.shape == ref_shape, f"{year} scene shape {s.shape} != {ref_shape}"
print(type(scenes["2018"]).__name__, ref_shape, scenes["2018"].crs)


## Zero-shot inference

The model checkpoint is the OSCD-trained `exp1` best — nothing is retrained. Three pairwise
passes: the two consecutive pairs tell you **when** change appeared; the full-span pair is
the headline **what changed by 2025** map. Windows stream from disk (chip 256, stride 256,
probability-averaged like the OSCD test maps).

In [ ]:
import time
from s2cd.data.dataset import load_stats_from_manifest
from s2cd.evaluate import load_best_model, predict_city

model, cfg, device = load_best_model("configs/train.yaml", store)
stats = load_stats_from_manifest(store.read_manifest())

PAIRS = [("2018", "2021"), ("2021", "2025"), ("2018", "2025")]
probs, infer_times = {}, {}
for a, b in PAIRS:
    t_infer = time.time()
    probs[f"{a}_{b}"] = predict_city(model, scenes[a], scenes[b], stats,
                                     chip=256, stride=256)
    infer_times[f"{a}_{b}"] = time.time() - t_infer
    print(f"{a} -> {b}: {ref_shape[1]}x{ref_shape[2]} px in {infer_times[f'{a}_{b}']:.1f}s")


## Figure conventions

Every panel is labeled with the **actual acquisition date** (e.g. "24 Jan 2018") — never
internal epoch names — so each figure explains itself. RGB is stretched to 0–0.3 reflectance;
change probability is on the magma sequential ramp pinned to the full 0–1 range, with a labeled
colorbar (probabilities are never read by hue alone).

In [ ]:
from datetime import datetime
import matplotlib.pyplot as plt
import numpy as np
from s2cd.utils import load_config

bands = load_config("configs/data.yaml")["bands"]
rgb = [bands.index("B04"), bands.index("B03"), bands.index("B02")]
h, w = probs["2018_2025"].shape
qh, qw = h // 8, w // 8               # decimated quicklooks for display

def pretty(iso):
    """'2018-01-24' -> '24 Jan 2018' — dates people can read aloud."""
    d = datetime.strptime(iso, "%Y-%m-%d")
    return f"{d.day} {d:%b} {d.year}"

DATE = {y: pretty(scene_info[y]["date"]) for y in EPOCHS}
CLOUD = {y: scene_info[y]["cloud_pct"] for y in EPOCHS}

# a[rgb] (NOT a[[rgb]]): rgb is a list, so double brackets nest it as a
# 2-D index array -> (1, 3, H, W) and the transpose below fails.
stretch = lambda a: np.clip(a[rgb] / 0.3, 0, 1).transpose(1, 2, 0)
quick = {y: scenes[y].read(out_shape=(qh, qw)) for y in EPOCHS}
rep = store.scratch / "reports"; rep.mkdir(exist_ok=True)


### Figure 1 — the headline city map

Left to right: Chennai in 2018, Chennai in 2025, and where the model says change happened
between those two dates.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 6))
axes[0].imshow(stretch(quick["2018"]))
axes[0].set_title(f"Chennai · {DATE['2018']}\n(cloud {CLOUD['2018']}%)")
axes[1].imshow(stretch(quick["2025"]))
axes[1].set_title(f"Chennai · {DATE['2025']}\n(cloud {CLOUD['2025']}%)")
im = axes[2].imshow(probs["2018_2025"], cmap="magma", vmin=0, vmax=1)
axes[2].set_title(f"Change probability\n{DATE['2018']} → {DATE['2025']}")
fig.colorbar(im, ax=axes[2], fraction=0.04, pad=0.02, label="Probability of change")
for ax in axes:
    ax.set_axis_off()
plt.tight_layout()
fig.savefig(rep / "map_chennai.png", dpi=150, bbox_inches="tight")
store.push_report("map_chennai.png")
plt.show()


### Figure 2 — when did the change happen?

The same probability scale across all three panels: bright pixels in the first panel changed
between 2018 and 2021; bright pixels in the second appeared between 2021 and 2025. The third
panel is the full 2018 → 2025 span (matches Figure 1). One shared colorbar for all panels.

In [ ]:
pairs = [("2018", "2021"), ("2021", "2025"), ("2018", "2025")]
fig, axes = plt.subplots(1, 3, figsize=(19, 6))
for ax, (a, b) in zip(axes, pairs):
    im = ax.imshow(probs[f"{a}_{b}"], cmap="magma", vmin=0, vmax=1)
    ax.set_title(f"Change: {DATE[a]} → {DATE[b]}")
    ax.set_axis_off()
fig.colorbar(im, ax=axes, fraction=0.04, pad=0.02, label="Probability of change")
plt.tight_layout()
fig.savefig(rep / "map_chennai_progression.png", dpi=150, bbox_inches="tight")
store.push_report("map_chennai_progression.png")
plt.show()


### Figure 3 — hotspot close-ups

The three 512-px cells with the highest mean change probability over the full span
(non-overlapping grid). Each row is one location, read left to right as *what was there in
2018, what was there in 2021, what is there in 2025, and what the model flagged* — so you can
see the construction actually happening and rule out clouds or sensor glitches.

In [ ]:
K, win = 3, 512
cells = [(probs["2018_2025"][y:y+win, x:x+win].mean(), y, x)
         for y in range(0, h - win + 1, win)
         for x in range(0, w - win + 1, win)]
top = sorted(cells, reverse=True)[:K]

fig, axes = plt.subplots(K, 4, figsize=(17, 4.4 * K))
for i, (mean_p, y, x) in enumerate(top):
    axes[i, 0].imshow(stretch(scenes["2018"][:, y:y+win, x:x+win]))
    axes[i, 1].imshow(stretch(scenes["2021"][:, y:y+win, x:x+win]))
    axes[i, 2].imshow(stretch(scenes["2025"][:, y:y+win, x:x+win]))
    axes[i, 3].imshow(probs["2018_2025"][y:y+win, x:x+win], cmap="magma", vmin=0, vmax=1)
    axes[i, 0].set_ylabel(f"hotspot {i+1}\nmean prob {mean_p:.2f}", rotation=0,
                          labelpad=56, va="center")
    print(f"hotspot {i+1}: mean change probability {mean_p:.3f} "
          f"@ ({x*10//1000} km, {y*10//1000} km from the NW corner)")
for ax, year in zip(axes[0], ["2018", "2021", "2025"]):
    ax.set_title(f"{DATE[year]}")
axes[0, 3].set_title(f"Change probability\n{DATE['2018']} → {DATE['2025']}")
for ax in axes.flat:
    ax.set_axis_off()
plt.tight_layout()
fig.savefig(rep / "map_chennai_hotspots.png", dpi=150, bbox_inches="tight")
store.push_report("map_chennai_hotspots.png")
plt.show()


## Export artifacts

Every pairwise probability map is written as a **georeferenced GeoTIFF** (same CRS/transform
as the source scene, so it opens in QGIS/Google Earth Pro), plus a machine-readable record —
enough to reproduce every number in the analysis. All land in `MyDrive/s2cd/reports/`.

In [ ]:
import json
import rasterio

profile = scenes["2018"].profile       # georeferencing from the source scene
profile.update(driver="GTiff", count=1, dtype="float32", compress="deflate")
for a, b in PAIRS:
    name = f"change_prob_chennai_{a}_{b}.tif"
    with rasterio.open(rep / name, "w", **profile) as dst:
        dst.write(probs[f"{a}_{b}"], 1)
    store.push_report(name)

record = {
    "scope": "zero-shot transfer: OSCD-trained model applied unchanged; no Chennai labels",
    "model_run": cfg["run_name"],
    "aoi_wsen": CHENNAI_RECT,
    "bands": bands,
    "chip": 256, "stride": 256,
    "epochs": {y: {k: scene_info[y][k] for k in ("id", "date", "cloud_pct")}
                for y in EPOCHS},
    "pairs": {f"{a}->{b}": {"inference_seconds": round(infer_times[f"{a}_{b}"], 1),
                               "prob_shape": [int(h), int(w)]}
               for a, b in PAIRS},
    "device": describe_device(device),
}
with open(rep / "chennai_results.json", "w") as f:
    json.dump(record, f, indent=1)
store.push_report("chennai_results.json")
print("pushed: map_chennai.png, map_chennai_progression.png, "
      "map_chennai_hotspots.png, 3x change_prob_chennai_*.tif, "
      "chennai_results.json -> Drive")


## Interpretation notes (plain language)

- **What counts as signal:** spatially coherent high-probability clusters that line up with
visible construction in the image triple — bare land turning into buildings, or demolition.
The hotspot figure is where you check this.
- **What to discount:** residual cloud or shadow in any single date — same-season pairing
keeps vegetation out of the picture, but clouds are per-scene.
- **Changes that cancel out** (built in 2019, demolished in 2024) are invisible to the
2018 → 2025 map; only the consecutive-period panels can time-step what stayed.
- **No ground truth here:** Chennai has no labeled change dataset, so this is a qualitative
generalization result, not a scored benchmark — the scored benchmark stays on OSCD test cities.
- **Reproducibility:** scene IDs, dates, cloud %, AOI and timing all live in `chennai_results.json`;
re-run with a different AOI by changing `CHENNAI_RECT` and re-running the export cells.